In [0]:
WITH all_trips AS (
    SELECT * FROM yellow_tripdata_january
    UNION ALL
    SELECT * FROM yellow_tripdata_february
    UNION ALL
    SELECT * FROM yellow_tripdata_march
),

weather_flags AS (
    SELECT
        DATE                                                    AS weather_date,
        PRCP,
        CASE
            WHEN PRCP > 25 OR COALESCE(WT16, 0) = 1 THEN 1
            ELSE 0
        END                                                     AS is_rainy_day
    FROM `ny-weather-data`
),

clean_trips AS (
    SELECT
        t.tpep_pickup_datetime,
        t.PULocationID,
        t.DOLocationID,
        t.passenger_count,
        t.fare_amount,
        t.total_amount,
        DATE(t.tpep_pickup_datetime)                            AS pickup_date,
        w.is_rainy_day,
        TIMESTAMP_SECONDS(
            FLOOR(UNIX_TIMESTAMP(t.tpep_pickup_datetime) / 240) * 240
        )                                                       AS time_bucket_15min
    FROM all_trips t
    INNER JOIN weather_flags w
        ON DATE(t.tpep_pickup_datetime) = w.weather_date
    WHERE
        t.fare_amount       > 0
        AND t.total_amount  > 0
        AND t.trip_distance > 0
        AND t.passenger_count BETWEEN 1 AND 6
),

shareable_groups AS (
    SELECT
        pickup_date,
        is_rainy_day,
        PULocationID,
        DOLocationID,
        time_bucket_15min,
        COUNT(*)                                                AS trips_in_group,
        AVG(total_amount)                                       AS avg_total_per_trip
    FROM clean_trips
    GROUP BY
        pickup_date,
        is_rainy_day,
        PULocationID,
        DOLocationID,
        time_bucket_15min
    HAVING COUNT(*) >= 2
),

-- Apply the tiered discount model to every shareable group
savings_calculated AS (
    SELECT
        pickup_date,
        is_rainy_day,
        trips_in_group,
        avg_total_per_trip,

        -- What each person currently pays (solo trip, no sharing)
        avg_total_per_trip                                      AS solo_cost_per_person,

        -- What each person would pay after the rideshare discount
        CASE
            WHEN trips_in_group = 2  THEN avg_total_per_trip * 0.60
            WHEN trips_in_group = 3  THEN avg_total_per_trip * 0.45
            ELSE                          avg_total_per_trip * 0.35
        END                                                     AS shared_cost_per_person,

        -- Dollar saving per individual passenger
        CASE
            WHEN trips_in_group = 2  THEN avg_total_per_trip * 0.40
            WHEN trips_in_group = 3  THEN avg_total_per_trip * 0.55
            ELSE                          avg_total_per_trip * 0.65
        END                                                     AS savings_per_person,

        -- Total savings for the whole group combined:
        -- per-person saving × number of people sharing
        CASE
            WHEN trips_in_group = 2  THEN avg_total_per_trip * 0.40 * trips_in_group
            WHEN trips_in_group = 3  THEN avg_total_per_trip * 0.55 * trips_in_group
            ELSE                          avg_total_per_trip * 0.65 * trips_in_group
        END                                                     AS total_group_savings_usd
    FROM shareable_groups
)

SELECT
    CASE WHEN is_rainy_day = 1 THEN 'Rainy Day' ELSE 'Dry Day'
    END                                                         AS weather_type,
    COUNT(*)                                                    AS shareable_groups,
    SUM(trips_in_group)                                         AS total_trips_involved,
    ROUND(AVG(solo_cost_per_person),   2)                       AS avg_solo_fare_usd,
    ROUND(AVG(shared_cost_per_person), 2)                       AS avg_shared_fare_usd,
    ROUND(AVG(savings_per_person),     2)                       AS avg_savings_per_person_usd,
    ROUND(
        AVG(savings_per_person / solo_cost_per_person * 100), 1
    )                                                           AS avg_savings_pct,
    -- Total dollars saved across all shareable trips in the dataset
    ROUND(SUM(total_group_savings_usd), 2)                      AS total_potential_savings_usd,
    -- Average daily savings: the headline number for a business case
    ROUND(
        SUM(total_group_savings_usd) / COUNT(DISTINCT pickup_date), 2
    )                                                           AS avg_daily_savings_usd,
    -- Breakdown: where does the saving come from?
    ROUND(SUM(CASE WHEN trips_in_group = 2
              THEN total_group_savings_usd ELSE 0 END), 2)      AS savings_from_pairs_usd,
    ROUND(SUM(CASE WHEN trips_in_group = 3
              THEN total_group_savings_usd ELSE 0 END), 2)      AS savings_from_trios_usd,
    ROUND(SUM(CASE WHEN trips_in_group >= 4
              THEN total_group_savings_usd ELSE 0 END), 2)      AS savings_from_4plus_usd
FROM savings_calculated
GROUP BY is_rainy_day
ORDER BY is_rainy_day DESC;